# S02-06 Importing OBJ Files - API Flow

This refactored workflow responds to the feedback by using TopologicPy's built-in API instead of manually constructing adjacency and access graphs. The aim is to keep the process reusable for other buildings.

## 1. Import the needed libraries

In [ ]:
from collections import defaultdict

from topologicpy.Topology import Topology
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Vertex import Vertex
from topologicpy.Helper import Helper

## 2. Check the TopologicPy version and renderer

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

renderer = "vscode"

## 3. Import the OBJ file and create a CellComplex

The OBJ is imported as a topology. The cells are then collected into a `CellComplex`, which is the topology that will be passed directly to TopologicPy's graph generator.

In [ ]:
obj_path = r"C:\Users\Arq. David Agudelo\GRAPHML_DAVID_AGUDELO_2026\Test_04.obj"

model = Topology.ByOBJPath(obj_path, selfMerge=True)[0]
all_cells = Topology.Cells(model)

cells_by_volume = sorted(all_cells, key=lambda c: Cell.Volume(c), reverse=True)
volumes = [Cell.Volume(c) for c in cells_by_volume]
split = max(range(len(volumes) - 1), key=lambda i: volumes[i] / volumes[i + 1]) + 1

room_cells = cells_by_volume[:split]
door_boxes = cells_by_volume[split:]
house = CellComplex.ByCells(room_cells)

print("Imported type:", Topology.TypeAsString(model))
print("Rooms/spaces detected:", len(room_cells))
print("Door aperture boxes detected:", len(door_boxes))
print("House type:", Topology.TypeAsString(house))

## 4. Create one semantic point per room and per door aperture

`InternalVertex` is used instead of `Centroid` because concave rooms, such as L-shaped spaces, can have centroids outside their geometry. Door apertures are imported as boxes, so connected boxes are grouped and represented by one blue point per logical door.

In [ ]:
def styled_vertex(vertex, label, category, color, size):
    dictionary = Dictionary.ByKeysValues(
        ["label", "category", "color", "size"],
        [label, category, color, size]
    )
    return Topology.SetDictionary(vertex, dictionary)


def average_vertex(vertices):
    x = sum(Vertex.X(v) for v in vertices) / len(vertices)
    y = sum(Vertex.Y(v) for v in vertices) / len(vertices)
    z = sum(Vertex.Z(v) for v in vertices) / len(vertices)
    return Vertex.ByCoordinates(x, y, z)


door_ids = {all_cells.index(cell) for cell in door_boxes}
door_adjacency = defaultdict(set)

for cell_id in door_ids:
    neighbors = Topology.AdjacentTopologies(all_cells[cell_id], model, topologyType="cell") or []
    for neighbor in neighbors:
        for other_id in door_ids:
            if Topology.IsSame(neighbor, all_cells[other_id]):
                door_adjacency[cell_id].add(other_id)
                break

door_groups = []
seen = set()
for cell_id in sorted(door_ids):
    if cell_id in seen:
        continue
    stack = [cell_id]
    seen.add(cell_id)
    group = []
    while stack:
        current = stack.pop()
        group.append(all_cells[current])
        for neighbor in door_adjacency[current]:
            if neighbor not in seen:
                seen.add(neighbor)
                stack.append(neighbor)
    door_groups.append(group)

room_vertices = [
    styled_vertex(Topology.InternalVertex(cell), f"Room {i}", "room", "red", 18)
    for i, cell in enumerate(room_cells, start=1)
]

door_vertices = [
    styled_vertex(
        average_vertex([Topology.InternalVertex(cell) for cell in group]),
        f"Door {i}",
        "door_aperture",
        "blue",
        14
    )
    for i, group in enumerate(door_groups, start=1)
]

door_apertures = [Cluster.ByTopologies(group) for group in door_groups]
house = Topology.AddApertures(house, door_apertures)
house = Topology.TransferDictionariesBySelectors(house, room_vertices, tranCells=True, tolerance=0.001)

point_graph = Graph.ByVerticesEdges(room_vertices + door_vertices, [])

print("Room points:", len(room_vertices))
print("Door aperture boxes:", len(door_boxes))
print("Logical door aperture points:", len(door_vertices))
print("Registered apertures:", len(Topology.Apertures(house) or []))

## 5. Generate the graph automatically

`Graph.ByTopology` creates the room graph from the `CellComplex`. The separate `point_graph` keeps one visible point per room and one visible point per logical door aperture.

In [ ]:
room_graph = Graph.ByTopology(
    house,
    direct=True,
    useInternalVertex=True
)

print("Room graph vertices:", len(Graph.Vertices(room_graph)))
print("Room graph edges:", len(Graph.Edges(room_graph)))
print("Display points:", len(Graph.Vertices(point_graph)))

## 6. Visualize the CellComplex and graph

In [ ]:
Topology.Show(
    house,
    room_graph,
    point_graph,
    vertexLabelKey="label",
    showVertexLabel=True,
    vertexColorKey="color",
    vertexSizeKey="size",
    faceOpacity=0.15,
    width=900,
    height=650,
    backgroundColor="white",
    renderer=renderer
)

## Summary

This version reduces the previous manual workflow to a compact API-based process:

1. Import libraries and configure the renderer.
2. Import the OBJ model.
3. Build a `CellComplex`.
4. Group door aperture boxes into one logical point per door.
5. Transfer semantic dictionaries through selectors.
6. Generate and visualize the graph with `Graph.ByTopology`.

The result is less brittle because the graph is produced from the topology itself rather than from custom adjacency and edge-building logic.